In [14]:
import pymorphy3
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
import joblib  
import re      

In [ ]:
# Подготовим данные для обучения
def prepare_dataset(input_file, output_file):
    labeled_data = []
    
    with open(input_file, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line: continue
            
            # 1. Определяем метку на основе позиции символа ударения `
            # Метка 0: зАмок
            # Метка 1: замОк
            if 'за`мок' in line.lower():
                label = 0
            elif 'замо`к' in line.lower():
                label = 1
            else:
                continue # Пропускаем строки без ударений
            
            # 2. Очищаем текст от символа ударения
            clean_text = line.replace('`', '')
            
            # 3. Сохраняем в формате: Текст [TAB] Метка
            labeled_data.append(f"{clean_text}\t{label}")

    with open(output_file, 'w', encoding='utf-8') as f:
        f.write('\n'.join(labeled_data))
    
    print(f"Готово! Размечено примеров: {len(labeled_data)}")
    print(f"Файл сохранен как: {output_file}")

# Запуск
prepare_dataset('замок.test', 'dataset_ready.txt')


Готово! Размечено примеров: 878
Файл сохранен как: dataset_ready.txt


In [ ]:
import pymorphy3
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

morph = pymorphy3.MorphAnalyzer()

# Функция для приведения всех слов в предложении к начальной форме
def lemmatize(text):
    words = text.split()
    res = [morph.parse(word)[0].normal_form for word in words]
    return " ".join(res)

# 1. Загрузка и предобработка
X_raw, y = load_ready_data('dataset_ready.txt')
X_lemmatized = [lemmatize(t) for t in X_raw]

# 2. Обучение
vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(X_lemmatized)
model = LogisticRegression()
model.fit(X, y)

# 3. Функция предсказания
def predict_accent(sentence):
    # Лемматизируем входное предложение перед предсказанием
    clean_sentence = lemmatize(sentence)
    vector = vectorizer.transform([clean_sentence])
    prediction = model.predict(vector)[0]
    
    if prediction == 0:
        return f"Результат: зАмок (архитектура) в предложении: '{sentence}'"
    else:
        return f"Результат: замОк (механизм) в предложении: '{sentence}'"

# Теперь это должно сработать правильно:
# --- ТЕСТ ---
print(predict_accent("на горе возвышался старинный каменный замок"))
print(predict_accent("наш дверной замок сломался"))



Результат: зАмок (архитектура) в предложении: 'на горе возвышался старинный каменный замок'
Результат: замОк (механизм) в предложении: 'наш дверной замок сломался'


In [ ]:
import joblib

# Сохранение в один файл
joblib.dump({'vectorizer': vectorizer, 'model': model}, 'castle_vs_lock_model.joblib')

print("Модель успешно сохранена!")


Модель успешно сохранена!


In [24]:
# Проверка версий библиотек для requirements.txt

print(f"scikit-learn: {sklearn.__version__}")
print(f"pymorphy3: {pymorphy3.__version__}")
print(f"joblib: {joblib.__version__}")


scikit-learn: 1.8.0
pymorphy3: 2.0.6
joblib: 1.5.3
